In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import os
import sys
import dill
import pprint
from pathlib import Path
import pathos
sys.path.append('/Users/leonardo.tessarolo/git/bayesian_bss/')

import numpy as np
import functools
import pandas as pd
import scipy.stats as st
from numpyro.diagnostics import gelman_rubin

from matplotlib import pyplot as plt

from src import ExponentialPrior, LogisticSource, TriangularSource, StandardLogisticSource, StudentTSource, DiscreteExperimentExecutor, LoggingUtilities
np.random.seed(100)

In [2]:
# Whether or not to create config
EXECUTE=True

# Mode
MODE='narrow'

# Whether or not to create folder structure
CREATE_FOLDER_STRUCTURE=False

# Experiment name
EXPERIMENT_NAME='CBA_SBRT_complete_analysis'

# Folder which will contain output directory tree
OUTPUT_DIR='./output'
base_output_path=Path(OUTPUT_DIR)

# Create experiment directory
experiment_dir = base_output_path / EXPERIMENT_NAME

# Instantiate logger
LoggingUtilities.configure_logging(
    log_dir=experiment_dir,
    log_type='discrete'
)

# 1. Configurations

In [3]:
if EXECUTE:
    
    # Number of sources and observations
    NSOURCES=2
    NOBS=1000

    # Mixing matrix configuration
    A = np.array([
        [1, 1],
        [-0.5, 0.5]
    ])

    # Ideal separating matrix
    B = np.linalg.inv(A)
    if MODE=='narrow':
        # Execution configurations
        exec_configs = {
            'general': {
                # General output directory
                'base_output_path': base_output_path,
                # Experiment name for folder structure
                'experiment_name': EXPERIMENT_NAME,
                # Experiment directory
                'experiment_dir': experiment_dir,
                # Whether or not to create folder structure
                'create_folder_structure': CREATE_FOLDER_STRUCTURE,
                # Whether or not to save graphs
                'save_graphs': False,
                # Number of sources
                'n_sources': NSOURCES,
                # Number of observations in each realization
                'n_obs': NOBS,
                # Number of parallel workers
                'n_workers': os.cpu_count(),
                # Number of parallel realizations to run
                'n_realizations': 150,
                # Mixing matrix
                'A': A,
                # Separating matrix
                'B': B,
                # Initial condition
                'initial_B': B,
                # Whether or not to use a normalized posterior
                'normalize_posterior': True,
                # Whether or not to have a progress bar for each initialization test case
                'initialization_progress_bar': False,
                # Whether or not to have a progress bar for each realization
                'realizations_progress_bar': True
            },
            'initialization': {
                'strategy': 'gmm',
                'starting_value_experiment': 'CBA_SBRT_broad_analysis'
            },
            'contour': {
                'perfect_model':{
                    # Number of points used in contour line grids
                    'contour_grid_points': 401,
                    # 'contour_grid_points': 3,
                    # Exploration limits for grids
                    'u_lims':(-0.4, 0.4),
                    'v_lims':(-0.4, 0.4),
                    # Central point
                    'central_point': (0,0)
                },
                'slightly_misspecified_model':{
                    # Number of points used in contour line grids
                    'contour_grid_points': 401,
                    # 'contour_grid_points': 3,
                    # Exploration limits for grids
                    'u_lims':(-0.4, 0.4),
                    'v_lims':(-0.4, 0.4),
                    # Central point
                    'central_point': (0,0)
                },
                'largely_misspecified_model':{
                    # Number of points used in contour line grids
                    'contour_grid_points': 401,
                    # Exploration limits for grids
                    'u_lims':(-0.8, 0.0),
                    'v_lims':(0.7, 1.5),
                    # Central point
                    'central_point': (-0.4,1.1)
                },
            },
            'map': {
                # Threshold for stopping optimization
                'stopping_thresh': 1E-11,
                # Maximum number of iterations
                'max_it': 1500000,
                # Minimum iterations
                'min_it': 1000,
                # Learning rate
                'learning_rate': 1E-6,
                # Persistance iterations for stopping criterion
                'stopping_criterion_persistance_its': 20
            },
            'mcmc': {
                # Initial condition noise
                'starting_distribution': lambda: st.t.rvs(
                    loc=0,
                    scale=0.1,
                    df=3,
                    size=B.shape
                ),
                # Exploration variance for MCMC
                'exploration_var': 5E-4,
                # Number of samples to generate
                'n_samples': 30000,
                # Number of parallel chains for MCMC
                'parallel_chains': 4,
                # Maximum iterations for MCMC
                'max_it': 20000,
                # Convergence threshold for R-hat diagnostic
                'R_hat_thresh': 1.01,
                # Interval at which R-hat will be evaluated
                'R_hat_evaluation_step': 500,
                # Number of consecutive convergent evaluations necessary
                'R_hat_persistance': 2,
                # Samples which will be discarded
                'R_hat_minimum_burn_in': 2000,
                # Whether or not to adjust exploration variance inside MCMC
                'auto_adjust_exploration_var': True,
                # Whether or not to show progress bars
                'progress_bar': False,
                # Target acceptance probability, in case auto adjust is enabled
                'target_accept_prob': 0.5,

            },
            'sim': {
                'source_model': StandardLogisticSource(use_jax=True),
                'sources': {
                    'perfect_model': StandardLogisticSource(use_jax=True),
                    'slightly_misspecified_model': LogisticSource(
                        mu=0.1,
                        sigma=1.1,
                        use_jax=True
                    ),
                    'largely_misspecified_model': TriangularSource(
                        lower=-2,
                        upper=2,
                        mode=0,
                        use_jax=True
                    )
                },
                'priors': {
                    'likelihood': ExponentialPrior(
                        center=np.linalg.inv(A),
                        std=np.inf,
                        use_jax=True
                    ),
                    'non_informative_prior': ExponentialPrior(
                        center=np.linalg.inv(A),
                        std=10,
                        use_jax=True
                    ),
                    'informative_prior': ExponentialPrior(
                        center=np.linalg.inv(A),
                        std=0.1,
                        use_jax=True
                    ),
                    'identity_transform': ExponentialPrior(
                        center=np.eye(NSOURCES),
                        std=0.1,
                        use_jax=True
                    )
                },
                'test_cases': {
                    'i': {
                        'source': 'perfect_model',
                        'prior': 'likelihood',
                    },
                    'ii': {
                        'source': 'perfect_model',
                        'prior': 'non_informative_prior',
                    },
                    'iii': {
                        'source': 'perfect_model',
                        'prior': 'informative_prior',
                    },
                    'iv': {
                        'source': 'perfect_model',
                        'prior': 'identity_transform',
                    },
                    'v': {
                        'source': 'slightly_misspecified_model',
                        'prior': 'likelihood',
                    },
                    'vi': {
                        'source': 'slightly_misspecified_model',
                        'prior': 'non_informative_prior',
                    },
                    'vii': {
                        'source': 'slightly_misspecified_model',
                        'prior': 'informative_prior',
                    },
                    'viii': {
                        'source': 'slightly_misspecified_model',
                        'prior': 'identity_transform',
                    },
                    'ix': {
                        'source': 'largely_misspecified_model',
                        'prior': 'likelihood',
                    },
                    'x': {
                        'source': 'largely_misspecified_model',
                        'prior': 'non_informative_prior',
                    },
                    'xi': {
                        'source': 'largely_misspecified_model',
                        'prior': 'informative_prior',
                    },
                    'xii': {
                        'source': 'largely_misspecified_model',
                        'prior': 'identity_transform',
                    },
                }
            }
        }

    elif MODE=='broad':
            exec_configs = {
                'general': {
                    # General output directory
                    'base_output_path': base_output_path,
                    # Experiment name for folder structure
                    'experiment_name': EXPERIMENT_NAME,
                    # Experiment directory
                    'experiment_dir': experiment_dir,
                    # Whether or not to create folder structure
                    'create_folder_structure': CREATE_FOLDER_STRUCTURE,
                    # Whether or not to save graphs
                    'save_graphs': False,
                    # Number of sources
                    'n_sources': NSOURCES,
                    # Number of observations in each realization
                    'n_obs': NOBS,
                    # Number of parallel workers
                    'n_workers': os.cpu_count(),
                    # Number of parallel realizations to run
                    'n_realizations': 1,
                    # Mixing matrix
                    'A': A,
                    # Separating matrix
                    'B': B,
                    # Initial condition
                    'initial_B': B,
                    # Whether or not to use a normalized posterior
                    'normalize_posterior': True,
                    # Whether or not to have a progress bar for each initialization test case
                    'initialization_progress_bar': False,
                    # Whether or not to have a progress bar for each realization
                    'realizations_progress_bar': True
                },
                'initialization': {
                    'strategy': 'random',
                    'starting_value_experiment': 'posterior_broad_analysis_low_var_1M'
                },
                'contour': {
                    'perfect_model':{
                        # Number of points used in contour line grids
                        'contour_grid_points': 3,
                        # Exploration limits for grids
                        'u_lims':(-0.4, 0.4),
                        'v_lims':(-0.4, 0.4),
                        # Central point
                        'central_point': (0,0)
                    },
                    'slightly_misspecified_model':{
                        # Number of points used in contour line grids
                        'contour_grid_points': 3,
                        # Exploration limits for grids
                        'u_lims':(-0.4, 0.4),
                        'v_lims':(-0.4, 0.4),
                        # Central point
                        'central_point': (0,0)
                    },
                    'largely_misspecified_model':{
                        # Number of points used in contour line grids
                        'contour_grid_points': 3,
                        # Exploration limits for grids
                        'u_lims':(-0.8, 0.0),
                        'v_lims':(0.7, 1.5),
                        # Central point
                        'central_point': (-0.4,1.1)
                    },
                },
                'map': {
                    # Threshold for stopping optimization
                    'stopping_thresh': 1E-10,
                    # Maximum number of iterations
                    'max_it': 2,
                    # Minimum iterations
                    'min_it': 1,
                    # Learning rate
                    'learning_rate': 1E-5,
                    # Persistance iterations for stopping criterion
                    'stopping_criterion_persistance_its': 20
                },
                'mcmc': {
                    # Initial condition noise
                    'starting_distribution': lambda: st.t.rvs(
                        loc=0,
                        scale=10,
                        df=3,
                        size=B.shape
                    ),
                    # Exploration variance for MCMC
                    'exploration_var': 1E-4,
                    # Number of samples to generate
                    'n_samples': 100000,
                    # Number of parallel chains for MCMC
                    'parallel_chains': 100,
                    # Maximum iterations for MCMC
                    'max_it': 25000,
                    # Convergence threshold for R-hat diagnostic
                    'R_hat_thresh': 1.01,
                    # Interval at which R-hat will be evaluated
                    'R_hat_evaluation_step': 500,
                    # Number of consecutive convergent evaluations necessary
                    'R_hat_persistance': 2,
                    # Samples which will be discarded
                    'R_hat_minimum_burn_in': 10000,
                    # Whether or not to adjust exploration variance inside MCMC
                    'auto_adjust_exploration_var': False,
                    # Whether or not to show progress bars
                    'progress_bar': False,
                    # Target acceptance probability, in case auto adjust is enabled
                    'target_accept_prob': 0.5,

                },
                'sim': {
                    'source_model': StandardLogisticSource(use_jax=True),
                    'sources': {
                        'perfect_model': StandardLogisticSource(use_jax=True),
                        'slightly_misspecified_model': LogisticSource(
                            mu=0.1,
                            sigma=1.1,
                            use_jax=True
                        ),
                        'largely_misspecified_model': TriangularSource(
                            lower=-2,
                            upper=2,
                            mode=0,
                            use_jax=True
                        )
                    },
                    'priors': {
                        'likelihood': ExponentialPrior(
                            center=np.linalg.inv(A),
                            std=np.inf,
                            use_jax=True
                        ),
                        'non_informative_prior': ExponentialPrior(
                            center=np.linalg.inv(A),
                            std=10,
                            use_jax=True
                        ),
                        'informative_prior': ExponentialPrior(
                            center=np.linalg.inv(A),
                            std=0.1,
                            use_jax=True
                        ),
                        'identity_transform': ExponentialPrior(
                            center=np.eye(NSOURCES),
                            std=0.1,
                            use_jax=True
                        )
                    },
                    'test_cases': {
                        'i': {
                            'source': 'perfect_model',
                            'prior': 'likelihood',
                        },
                        'ii': {
                            'source': 'perfect_model',
                            'prior': 'non_informative_prior',
                        },
                        'iii': {
                            'source': 'perfect_model',
                            'prior': 'informative_prior',
                        },
                        'iv': {
                            'source': 'perfect_model',
                            'prior': 'identity_transform',
                        },
                        'v': {
                            'source': 'slightly_misspecified_model',
                            'prior': 'likelihood',
                        },
                        'vi': {
                            'source': 'slightly_misspecified_model',
                            'prior': 'non_informative_prior',
                        },
                        'vii': {
                            'source': 'slightly_misspecified_model',
                            'prior': 'informative_prior',
                        },
                        'viii': {
                            'source': 'slightly_misspecified_model',
                            'prior': 'identity_transform',
                        },
                        'ix': {
                            'source': 'largely_misspecified_model',
                            'prior': 'likelihood',
                        },
                        'x': {
                            'source': 'largely_misspecified_model',
                            'prior': 'non_informative_prior',
                        },
                        'xi': {
                            'source': 'largely_misspecified_model',
                            'prior': 'informative_prior',
                        },
                        'xii': {
                            'source': 'largely_misspecified_model',
                            'prior': 'identity_transform',
                        },
                    }
                }
            }
else:
    with (experiment_dir/'execution_config.pkl').open('rb') as f:
        exec_configs = dill.load(f)

pprint.pp(exec_configs)

{'general': {'base_output_path': PosixPath('output'),
             'experiment_name': 'CBA_SBRT_complete_analysis',
             'experiment_dir': PosixPath('output/CBA_SBRT_complete_analysis'),
             'create_folder_structure': False,
             'save_graphs': False,
             'n_sources': 2,
             'n_obs': 1000,
             'n_workers': 8,
             'n_realizations': 150,
             'A': array([[ 1. ,  1. ],
       [-0.5,  0.5]]),
             'B': array([[ 0.5, -1. ],
       [ 0.5,  1. ]]),
             'initial_B': array([[ 0.5, -1. ],
       [ 0.5,  1. ]]),
             'normalize_posterior': True,
             'initialization_progress_bar': False,
             'realizations_progress_bar': True},
 'initialization': {'strategy': 'gmm',
                    'starting_value_experiment': 'CBA_SBRT_broad_analysis'},
 'contour': {'perfect_model': {'contour_grid_points': 401,
                               'u_lims': (-0.4, 0.4),
                               'v_li

# 2. Folder Structure

In [4]:
# Initialize folder structure, if so specified
if CREATE_FOLDER_STRUCTURE:
    if exec_configs['general']['create_folder_structure']:
        # Creates base output path and experiment dir
        if not base_output_path.is_dir():
            base_output_path.mkdir()
        if not experiment_dir.is_dir():
            experiment_dir.mkdir()

        # Creates initialization results folder
        initialization_dir = experiment_dir / 'initial_mode_seeking'
        if not initialization_dir.is_dir():
            initialization_dir.mkdir()
        # for r in range(exec_configs['initial_mode_seeking']['n_realizations']):
        #     # Overall folder for realization
        #     realization_dir = initialization_dir / str(r)
        #     if not realization_dir.is_dir():
        #         realization_dir.mkdir()

        # Creates folders for individual realizations
        for r in range(exec_configs['general']['n_realizations']):
            # Overall folder for realization
            realization_dir = experiment_dir / str(r)
            if not realization_dir.is_dir():
                realization_dir.mkdir()
            # Initialize success flag
            with (realization_dir/'success_flag.pkl').open('wb') as f:
                    dill.dump('UNFINISHED', f)
        

if EXECUTE:
     with (experiment_dir/'execution_config.pkl').open('wb') as f:
        dill.dump(exec_configs, f)


# 3. Execute Experiment

In [5]:
%%time
if EXECUTE:
    ex = DiscreteExperimentExecutor(
        cfg=exec_configs,
        # initialize=CREATE_FOLDER_STRUCTURE
    )
    ex.run()

####################################################################################################
Total realizations: 150
Finished realizations: 121
Unfinished realizations: 29
####################################################################################################
Executando experimento


100%|██████████| 29/29 [9:26:35<00:00, 1172.26s/it]  

CPU times: user 14h 59min 48s, sys: 1h 3min 54s, total: 16h 3min 43s
Wall time: 9h 26min 35s
